In [1]:
tabla='ctaam10'
col_tabla = "atenambatenfec"
essi='essi_dat_cex002_etl'
col_essi='ate_fec'
dat= "dat_cext002_essi"
col_dat='ate_fec'

In [2]:
from decouple import config
import decouple #eliminar en .py
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [14]:
from datetime import datetime
from sqlalchemy import text


#fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=16", con=connection2)
#fecha= fecha.iloc[0, 0]
#print(fecha)
#
now = datetime.now()
#
#query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=16"
#
#c1= text(query)
#connection2.execute(c1)
fecha="2023-10-23"

In [57]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

In [18]:
base1.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'ate_num',
       'ate_fec', 'tip_con', 'des_tip', 'cod_ser', 'des_ser', 'cod_cps',
       'des_cps', 'ate_cod', 'des_ate', 'ley_cod', 'des_ley', 'res_cod',
       'des_res', 'est_reg', 'des_est', 'usu_cre', 'fec_cre', 'usu_mod',
       'fec_mod', 'fec_ult_reg', 'cod_tdi', 'num_doc', 'pac_sec',
       'cond_diagcod', 'des_condiag', 'diag_cod', 'des_diag', 'diag_ord',
       'tip_diag', 'des_tipdiag', 'caso_diag', 'caso_diagdes', 'alta_diag'],
      dtype='object')

In [19]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236973 entries, 0 to 236972
Data columns (total 39 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   ori_cas       236954 non-null  object             
 1   cod_cas       236954 non-null  object             
 2   des_cas       236954 non-null  object             
 3   cod_red       236954 non-null  object             
 4   des_red       236954 non-null  object             
 5   ate_num       236954 non-null  float64            
 6   ate_fec       236973 non-null  datetime64[ns, UTC]
 7   tip_con       236973 non-null  object             
 8   des_tip       236973 non-null  object             
 9   cod_ser       0 non-null       object             
 10  des_ser       0 non-null       object             
 11  cod_cps       236973 non-null  object             
 12  des_cps       236973 non-null  object             
 13  ate_cod       0 non-null       object       

In [10]:
base1.shape

(0, 39)

In [11]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

#TRAEMOS TODOS LOS MAESTROS

In [12]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
#id_red,id_cas

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

tipcon=pd.read_sql_query(f"SELECT id_tipcon,cod_tipcon FROM dim_tipcon", con=connection2)
tipcon['cod_tipcon']=tipcon['cod_tipcon'].str.strip()

servicios = pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
cps['cod_cps']=cps['cod_cps'].str.strip()

numaten = pd.read_sql_query(f"SELECT id_numaten,cod_numaten FROM dim_numaten", con=connection2)
numaten['cod_numaten']=numaten['cod_numaten'].str.strip()

tipleycont = pd.read_sql_query(f"SELECT id_tipleycont,cod_tipleycont FROM dim_tipleycont", con=connection2)
tipleycont['cod_tipleycont']=tipleycont['cod_tipleycont'].str.strip()

resaten = pd.read_sql_query(f"SELECT id_resaten,cod_resaten FROM dim_resaten", con=connection2)
resaten['cod_resaten']=resaten['cod_resaten'].str.strip()

estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
estreg['cod_est']=estreg['cod_est'].str.strip()

personal = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
personal["num_doc"]=personal["num_doc"].str.strip()
personal["num_doc"]=personal["num_doc"].str.replace(' ', '', regex=True)
personal=personal.drop_duplicates(subset="num_doc")

#pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)

condiag = pd.read_sql_query(f"SELECT id_condia,cod_con FROM dim_condiag", con=connection2)
condiag['cod_con']=condiag['cod_con'].str.strip()

cie10 = pd.read_sql_query(f"SELECT id_cie,cod_cie FROM dim_cie10", con=connection2)
cie10['cod_cie']=cie10['cod_cie'].str.strip()

tipdx = pd.read_sql_query(f"SELECT id_tipdx,cod_tdx FROM dim_tipdx", con=connection2)
tipdx['cod_tdx']=tipdx['cod_tdx'].str.strip()

casdiag = pd.read_sql_query(f"SELECT id_casdiag,cod_casdiag FROM dim_casdiag", con=connection2)
casdiag['cod_casdiag']=casdiag['cod_casdiag'].str.strip()


In [20]:
#Eliminamos las columnas que no se usarán aquí: las descripciones y otras 4 más, previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'des_tip', 'des_ser', 'des_cps', 'des_ate',
                       'des_ley', 'des_res', 'des_est', 'cod_tdi', 'num_doc', 'des_condiag',
                       'des_diag', 'des_tipdiag', 'caso_diagdes']


# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)
base1.shape

(236973, 24)

In [21]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236973 entries, 0 to 236972
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   ori_cas       236954 non-null  object             
 1   cod_cas       236954 non-null  object             
 2   cod_red       236954 non-null  object             
 3   ate_num       236954 non-null  float64            
 4   ate_fec       236973 non-null  datetime64[ns, UTC]
 5   tip_con       236973 non-null  object             
 6   cod_ser       0 non-null       object             
 7   cod_cps       236973 non-null  object             
 8   ate_cod       0 non-null       object             
 9   ley_cod       236973 non-null  object             
 10  res_cod       236973 non-null  object             
 11  est_reg       236973 non-null  object             
 12  usu_cre       236973 non-null  object             
 13  fec_cre       236973 non-null  datetime64[ns

In [22]:
control_a.append(base1.shape[0])

In [23]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1=pd.merge(base1,oricas,how='left',on='ori_cas')
base1.shape

(236973, 25)

In [24]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [25]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='inner',on='cod_cas')
base1=pd.merge(base1,cas,how='left',on='cod_cas')
base1.shape

(236973, 25)

In [26]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [27]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='inner',on='cod_red')
base1=pd.merge(base1,red,how='left',on='cod_red')
base1.shape

(236973, 25)

In [28]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [29]:
base1['tip_con']=base1['tip_con'].str.strip()
base1["tip_con"]=base1["tip_con"].str.replace(' ', '', regex=True)
tipcon=tipcon.rename(columns={"cod_tipcon": "tip_con"})
merge=pd.merge(base1,tipcon,how='inner',on='tip_con')
base1=pd.merge(base1,tipcon,how='left',on='tip_con')
base1.shape

(236973, 25)

In [30]:
log_falla('id_tipcon', 'tip_con', True)
log_control('dim_tipcon')
base1=base1.drop('tip_con',axis=1)

In [31]:
base1['cod_ser']=base1['cod_ser'].str.strip()
base1["cod_ser"]=base1["cod_ser"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,servicios,how='inner',on='cod_ser')
base1=pd.merge(base1,servicios,how='left',on='cod_ser')
base1.shape

(236973, 25)

In [32]:
log_falla('id_serv','cod_ser', True)
log_control('dim_servicios')
base1 = base1.drop("cod_ser",axis=1)

In [33]:
base1['cod_cps']=base1['cod_cps'].str.strip()
base1["cod_cps"]=base1["cod_cps"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cps,how='inner',on='cod_cps')
base1=pd.merge(base1,cps,how='left',on='cod_cps')
base1.shape

(236973, 25)

In [34]:
log_falla('id_cps','cod_cps', True)
log_control('dim_cps')
base1 = base1.drop("cod_cps",axis=1)

In [35]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236973 entries, 0 to 236972
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   ate_num       236954 non-null  float64            
 1   ate_fec       236973 non-null  datetime64[ns, UTC]
 2   ate_cod       0 non-null       object             
 3   ley_cod       236973 non-null  object             
 4   res_cod       236973 non-null  object             
 5   est_reg       236973 non-null  object             
 6   usu_cre       236973 non-null  object             
 7   fec_cre       236973 non-null  datetime64[ns, UTC]
 8   usu_mod       127119 non-null  object             
 9   fec_mod       236973 non-null  datetime64[ns, UTC]
 10  fec_ult_reg   3571 non-null    object             
 11  pac_sec       236973 non-null  float64            
 12  cond_diagcod  236954 non-null  object             
 13  diag_cod      236954 non-null  object       

In [36]:
base1['ate_cod']=base1['ate_cod'].str.strip()
base1["ate_cod"]=base1["ate_cod"].str.replace(' ', '', regex=True)
numaten=numaten.rename(columns={"id_numaten": "id_aten","cod_numaten":"ate_cod"})
merge=pd.merge(base1,numaten,how='inner',on='ate_cod')
base1=pd.merge(base1,numaten,how='left',on='ate_cod')
base1.shape

(236973, 25)

In [37]:
log_falla('id_aten','ate_cod', True)
log_control('dim_numaten')
base1 = base1.drop("ate_cod",axis=1)

In [38]:
base1['ley_cod']=base1['ley_cod'].str.strip()
base1["ley_cod"]=base1["ley_cod"].str.replace(' ', '', regex=True)
tipleycont=tipleycont.rename(columns={"cod_tipleycont":"ley_cod"})
merge=pd.merge(base1,tipleycont,how='inner',on='ley_cod')
base1=pd.merge(base1,tipleycont,how='left',on='ley_cod')
base1.shape

(236973, 25)

In [39]:
log_falla('id_tipleycont','ley_cod', True)
log_control('dim_tipleycont')
base1 = base1.drop("ley_cod",axis=1)

In [40]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236973 entries, 0 to 236972
Data columns (total 24 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   ate_num        236954 non-null  float64            
 1   ate_fec        236973 non-null  datetime64[ns, UTC]
 2   res_cod        236973 non-null  object             
 3   est_reg        236973 non-null  object             
 4   usu_cre        236973 non-null  object             
 5   fec_cre        236973 non-null  datetime64[ns, UTC]
 6   usu_mod        127119 non-null  object             
 7   fec_mod        236973 non-null  datetime64[ns, UTC]
 8   fec_ult_reg    3571 non-null    object             
 9   pac_sec        236973 non-null  float64            
 10  cond_diagcod   236954 non-null  object             
 11  diag_cod       236954 non-null  object             
 12  diag_ord       236954 non-null  object             
 13  tip_diag       236954 non-nul

In [41]:
base1['res_cod']=base1['res_cod'].str.strip()
base1["res_cod"]=base1["res_cod"].str.replace(' ', '', regex=True)
resaten=resaten.rename(columns={"cod_resaten":"res_cod"})
merge=pd.merge(base1,resaten,how='inner',on='res_cod')
base1=pd.merge(base1,resaten,how='left',on='res_cod')
base1.shape

(236973, 25)

In [42]:
log_falla('id_resaten','res_cod', True)
log_control('dim_resaten')
base1 = base1.drop("res_cod",axis=1)

In [44]:
base1['est_reg']=base1['est_reg'].str.strip()
base1["est_reg"]=base1["est_reg"].str.replace(' ', '', regex=True)
estreg=estreg.rename(columns={"cod_est":"est_reg"})
merge=pd.merge(base1,estreg,how='inner',on='est_reg')
base1=pd.merge(base1,estreg,how='left',on='est_reg')
base1.shape

(236973, 25)

In [45]:
log_falla('id_estreg','est_reg', True)
log_control('dim_estreg')
base1 = base1.drop("est_reg",axis=1)

In [46]:
personal=personal.rename(columns={"num_doc": "usu_cre","id_person": "id_usucre"})
base1['usu_cre']=base1['usu_cre'].str.strip()
base1["usu_cre"]=base1["usu_cre"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,personal,how='inner',on='usu_cre')
base1=pd.merge(base1,personal,how='left',on='usu_cre') #Se perdieron unos datos
base1.shape

(236973, 25)

In [47]:
log_falla('id_usucre', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [48]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236973 entries, 0 to 236972
Data columns (total 24 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   ate_num        236954 non-null  float64            
 1   ate_fec        236973 non-null  datetime64[ns, UTC]
 2   fec_cre        236973 non-null  datetime64[ns, UTC]
 3   usu_mod        127119 non-null  object             
 4   fec_mod        236973 non-null  datetime64[ns, UTC]
 5   fec_ult_reg    3571 non-null    object             
 6   pac_sec        236973 non-null  float64            
 7   cond_diagcod   236954 non-null  object             
 8   diag_cod       236954 non-null  object             
 9   diag_ord       236954 non-null  object             
 10  tip_diag       236954 non-null  object             
 11  caso_diag      236954 non-null  object             
 12  alta_diag      236954 non-null  object             
 13  id_oricas      236954 non-nul

In [49]:
b=base1.copy()

In [50]:
base1["usu_mod"].unique()

array(['09563306  ', '73011999  ', '43418323  ', ..., '03489503  ',
       '72633763  ', '44481814  '], dtype=object)

In [51]:
personal=personal.rename(columns={"usu_cre": "usu_mod","id_usucre": "id_usumod"})
base1['usu_mod']=base1['usu_mod'].str.strip()
base1["usu_mod"]=base1["usu_mod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,personal,how='inner',on='usu_mod')
base1=pd.merge(base1,personal,how='left',on='usu_mod')
base1.shape

(236973, 25)

In [52]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [53]:
basex=base1.copy()

In [48]:
base1=basex

In [51]:
#Preparamos la variable y creamos una temporal para hacer el emparejamiento
# base1["pac_sec"]=base1["pac_sec"].astype('Int64')
# base1["temp_pac_sec"]=base1["pac_sec"].astype(str)
# pacsec["per_sec"]=pacsec["per_sec"].astype('Int64')
# pacsec["per_sec"]=pacsec["per_sec"].astype(str)

In [52]:
pacsec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23792262 entries, 0 to 23792261
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   id_pacsec     int64 
 1   temp_pac_sec  object
dtypes: int64(1), object(1)
memory usage: 363.0+ MB


In [53]:
pacsec=pacsec.rename(columns={"per_sec": "temp_pac_sec"})
pacsec['temp_pac_sec']=pacsec['temp_pac_sec'].str.strip()
pacsec["temp_pac_sec"]=pacsec["temp_pac_sec"].str.replace(' ', '', regex=True)
base1['temp_pac_sec']=base1['temp_pac_sec'].str.strip()
base1["temp_pac_sec"]=base1["temp_pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='temp_pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='temp_pac_sec')
base1=base1.drop('temp_pac_sec',axis=1)
base1.shape

(341029, 25)

In [54]:
log_falla('id_pacsec', 'pac_sec', True)
log_control('dim_pacsec')
base1=base1.drop('pac_sec',axis=1)

In [54]:
condiag=condiag.rename(columns={"cod_con": "cond_diagcod"})
base1['cond_diagcod']=base1['cond_diagcod'].str.strip()
base1["cond_diagcod"]=base1["cond_diagcod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,condiag,how='inner',on='cond_diagcod')
base1=pd.merge(base1,condiag,how='left',on='cond_diagcod')
base1.shape

(236973, 25)

In [55]:
log_falla('id_condia', 'cond_diagcod', True)
log_control('dim_condiag')
base1=base1.drop('cond_diagcod',axis=1)

In [56]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236973 entries, 0 to 236972
Data columns (total 24 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   ate_num        236954 non-null  float64            
 1   ate_fec        236973 non-null  datetime64[ns, UTC]
 2   fec_cre        236973 non-null  datetime64[ns, UTC]
 3   fec_mod        236973 non-null  datetime64[ns, UTC]
 4   fec_ult_reg    3571 non-null    object             
 5   pac_sec        236973 non-null  float64            
 6   diag_cod       236954 non-null  object             
 7   diag_ord       236954 non-null  object             
 8   tip_diag       236954 non-null  object             
 9   caso_diag      236954 non-null  object             
 10  alta_diag      236954 non-null  object             
 11  id_oricas      236954 non-null  float64            
 12  id_cas         236954 non-null  float64            
 13  id_red         236954 non-nul

In [57]:
cie10=cie10.rename(columns={"cod_cie": "diag_cod"})
base1['diag_cod']=base1['diag_cod'].str.strip()
base1["diag_cod"]=base1["diag_cod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cie10,how='inner',on='diag_cod')
base1=pd.merge(base1,cie10,how='left',on='diag_cod')
base1.shape

(341033, 25)

In [58]:
log_falla('id_cie', 'diag_cod', True)
log_control('dim_cie10')
base1=base1.drop('diag_cod',axis=1)

In [59]:
tipdx=tipdx.rename(columns={"cod_tdx": "tip_diag"})
base1['tip_diag']=base1['tip_diag'].str.strip()
base1["tip_diag"]=base1["tip_diag"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,tipdx,how='inner',on='tip_diag')
base1=pd.merge(base1,tipdx,how='left',on='tip_diag')
base1.shape

(341033, 25)

In [60]:
log_falla('id_tipdx', 'tip_diag', True)
log_control('dim_tipdx')
base1=base1.drop('tip_diag',axis=1)

In [61]:
casdiag=casdiag.rename(columns={"cod_casdiag": "caso_diag"})
base1['caso_diag']=base1['caso_diag'].str.strip()
base1["caso_diag"]=base1["caso_diag"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,casdiag,how='inner',on='caso_diag')
base1=pd.merge(base1,casdiag,how='left',on='caso_diag')
base1.shape

(341033, 25)

In [62]:
log_falla('id_casdiag', 'caso_diag', True)
base1=base1.drop('caso_diag', axis=1)
dim.append('dim_casdiag')
control_d.append(base1.shape[0])

In [63]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [64]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [65]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [ ]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [ ]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

6027

In [ ]:
fecha_fin = "2024-12-31"

In [ ]:
#proceso = "DAT"
#cod_proceso = 13

proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=16", con=connection2)
proceso = proceso.iloc[0, 0]
cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=16", con=connection2)
cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [ ]:
tabla_logs

In [ ]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

In [ ]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

In [ ]:
connection1.close()
connection2.close()
connection3.close()

In [ ]:
engine1.dispose()
engine2.dispose()
engine3.dispose()